# Google Search Ranking & Discoverability Capstone

**Research question:** Which page-level features are most associated with stronger Google search rankings?

This notebook:
1. Loads a CSV or Parquet dataset.
2. Detects a likely search-position target column.
3. Selects numerical features.
4. Runs correlation analysis.
5. Trains a Random Forest baseline model.
6. Prints ranked feature importances and evaluation metrics.

Do not publish confidential data.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# CHANGE THIS to your actual file.
DATA_PATH = Path("../data/flyrank_warehouse.csv")
RANDOM_STATE = 42


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}\n"
        "Place your approved CSV or Parquet file in the data/ folder "
        "and update DATA_PATH in the previous cell."
    )

suffix = DATA_PATH.suffix.lower()
if suffix == ".csv":
    df = pd.read_csv(DATA_PATH)
elif suffix in {".parquet", ".pq"}:
    df = pd.read_parquet(DATA_PATH)
else:
    raise ValueError("Supported formats: .csv, .parquet, .pq")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
display(df.head())


In [ ]:
# Try to find a likely ranking-position target column.
candidate_targets = [
    "position", "avg_position", "average_position", "rank",
    "ranking_position", "search_position", "google_position"
]

lower_to_original = {c.lower().strip(): c for c in df.columns}
target_col = next((lower_to_original[c] for c in candidate_targets if c in lower_to_original), None)

if target_col is None:
    likely = [c for c in df.columns if any(k in c.lower() for k in ["position", "rank"])]
    print("Could not automatically identify the target column.")
    print("Possible target columns:", likely)
    raise ValueError(
        "Set target_col manually, for example: target_col = 'position'"
    )

print("Detected target column:", target_col)


In [ ]:
# Keep the target and usable numerical features.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

if target_col not in numeric_cols:
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

feature_cols = [c for c in numeric_cols if c != target_col]

# Drop IDs and near-constant features when obvious.
blocked_tokens = ["id", "timestamp", "unix"]
feature_cols = [
    c for c in feature_cols
    if not any(token == c.lower() or c.lower().endswith("_" + token) for token in blocked_tokens)
]

# Keep columns with at least 30% non-null values and more than one unique value.
feature_cols = [
    c for c in feature_cols
    if df[c].notna().mean() >= 0.30 and df[c].nunique(dropna=True) > 1
]

working = df[feature_cols + [target_col]].copy()
working = working.dropna(subset=[target_col])

print("Rows available:", len(working))
print("Features selected:", len(feature_cols))
print(feature_cols[:30])


In [ ]:
# Correlation analysis.
corr = (
    working[feature_cols + [target_col]]
    .corr(numeric_only=True)[target_col]
    .drop(target_col)
    .dropna()
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print("Top correlations with ranking position:")
display(corr.head(15).to_frame("correlation_with_position"))


In [ ]:
X = working[feature_cols]
y = working[target_col]

if len(working) < 20:
    raise ValueError("Not enough usable rows for a reliable train/test split.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=2
    ))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(f"MAE: {mae:.4f}")
print(f"R²:  {r2:.4f}")


In [ ]:
rf = model.named_steps["rf"]
importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("Top feature importances:")
display(importance.head(15).to_frame("feature_importance"))

print("\n=== COPY THESE RESULTS INTO index.html ===")
print(f"Rows analyzed: {len(working):,}")
print(f"Number of numerical features used: {len(feature_cols)}")
print(f"Model MAE: {mae:.4f}")
print(f"Model R²: {r2:.4f}")
print("\nTop 5 features:")
for i, (feature, score) in enumerate(importance.head(5).items(), start=1):
    print(f"{i}. {feature}: {score:.4f}")


## Interpretation guide

For Google ranking position, a lower number usually represents a better ranking.

Be careful when interpreting correlations:
- A **negative correlation** with position can mean higher feature values are associated with better rankings.
- A **positive correlation** with position can mean higher feature values are associated with worse rankings.

Model feature importance shows predictive usefulness, not whether the relationship is positive or negative and not causation.


In [ ]:
# Optional: Save a clean results table for your paper.
results = pd.DataFrame({
    "feature": importance.index,
    "feature_importance": importance.values,
    "correlation_with_position": corr.reindex(importance.index).values
})

output_path = Path("../feature_results.csv")
results.to_csv(output_path, index=False)
print("Saved:", output_path.resolve())
display(results.head(15))
